In [11]:
import requests
import pandas as pd
import time

def scrape_yahoo_bulk(target_count=1000):
    all_stocks = []
    # Yahoo typically allows 100 items per request
    offset = 0
    count_per_page = 100
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    while len(all_stocks) < target_count:
        # This URL targets the 'Most Active' stocks screener API
        url = f"https://query1.finance.yahoo.com/v1/finance/screener/predefined/saved?formatted=true&scrIds=most_actives&count={count_per_page}&offset={offset}"
        
        try:
            response = requests.get(url, headers=headers)
            data = response.json()
            
            # Navigate the JSON structure
            quotes = data['finance']['result'][0]['quotes']
            
            if not quotes:
                break

            for stock in quotes:
                all_stocks.append({
                    # CATEGORICAL COLUMNS
                    'Ticker': stock.get('symbol'),
                    'Company': stock.get('shortName'),
                    'Market': stock.get('market'),
                    'QuoteType': stock.get('quoteType'),
                    
                    # NUMERICAL COLUMNS
                    'Price': stock.get('regularMarketPrice', {}).get('raw', 0),
                    'Change': stock.get('regularMarketChange', {}).get('raw', 0),
                    'Volume': stock.get('regularMarketVolume', {}).get('raw', 0),
                    'MarketCap': stock.get('marketCap', {}).get('raw', 0)
                })

            print(f"Collected {len(all_stocks)} rows...")
            offset += count_per_page
            time.sleep(1) # Be nice to their servers!
            
        except Exception as e:
            print(f"Reached end or encountered error: {e}")
            break

    # Convert to a DataFrame for analysis
    df = pd.DataFrame(all_stocks)
    return df

# Execute
final_df = scrape_yahoo_bulk(1000)

# Save to CSV
final_df.to_csv("yahoo_finance_1000_rows.csv", index=False)
print("Saved to yahoo_finance_1000_rows.csv")

Collected 100 rows...
Collected 200 rows...
Collected 300 rows...
Collected 400 rows...
Collected 500 rows...
Collected 600 rows...
Collected 700 rows...
Collected 800 rows...
Collected 900 rows...
Collected 1000 rows...
Saved to yahoo_finance_1000_rows.csv
